# Phase 1: Setup and Sanity Check
YOLO26 vs YOLO11 Small-Object Benchmark on VisDrone

**Before running this notebook:**
- Kaggle notebook settings: enable a GPU accelerator (T4 x1 is enough for nano models).
- Kaggle notebook settings: Internet must be On if you are not attaching a Kaggle dataset, the auto-download needs it.
- This notebook does not do the real training run. It confirms the environment, the dataset, and both model families work end to end on a small slice of data, so Phase 2 does not waste GPU hours discovering a broken pipeline.

**Dataset options:**
- Auto-download (default in this notebook): Ultralytics' built-in `VisDrone.yaml` downloads and converts the official dataset on first use, about 2.3 GB.
- Kaggle-attached mirror: https://www.kaggle.com/datasets/evilspirit05/visdrone . This is a third-party upload, not officially maintained. [verify] its folder structure (`images/train`, `images/val`, `labels/train`, `labels/val`) before trusting it. If it does not match, use the auto-download path instead, it is guaranteed correct.

In [1]:
# Ultralytics ships YOLO26 and YOLO11 in the same package, one install covers both
!pip install -q ultralytics onnxruntime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 69.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 5.0 MB/s eta 0:00:00


In [2]:
"""Confirm GPU and package versions before spending any training time."""
import torch
import ultralytics


def check_environment() -> None:
    print(f"ultralytics version: {ultralytics.__version__}")
    print(f"torch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"GPU count: {torch.cuda.device_count()}")
    else:
        # training would silently fall back to CPU and take hours instead of minutes,
        # better to fail loudly here than discover it 40 minutes into phase 2
        raise RuntimeError("No GPU detected. Enable a GPU accelerator in Kaggle notebook settings.")


check_environment()

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
ultralytics version: 8.4.124
torch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
GPU count: 2


In [3]:
"""
Locate the VisDrone dataset. Tries a Kaggle-attached dataset first (faster,
no download), falls back to Ultralytics' built-in auto-download and conversion.
"""
from pathlib import Path

KAGGLE_INPUT = Path("/kaggle/input")


def find_attached_visdrone() -> Path | None:
    """Look for a Kaggle-attached VisDrone dataset with the expected YOLO layout."""
    if not KAGGLE_INPUT.exists():
        return None
    for candidate in KAGGLE_INPUT.iterdir():
        if "visdrone" in candidate.name.lower():
            has_train_images = (candidate / "images" / "train").exists()
            has_val_images = (candidate / "images" / "val").exists()
            if has_train_images and has_val_images:
                return candidate
    return None


attached = find_attached_visdrone()
if attached:
    print(f"Found attached VisDrone dataset at: {attached}")
    print("Structure looks correct (images/train, images/val present).")
    print("If you want to use it, point data= at a yaml file with path set to this directory.")
else:
    print("No correctly structured attached dataset found.")
    print("Will use Ultralytics' built-in VisDrone.yaml auto-download in the next cell.")
    print("Requires Internet: On in notebook settings, downloads about 2.3 GB once.")

No correctly structured attached dataset found.
Will use Ultralytics' built-in VisDrone.yaml auto-download in the next cell.
Requires Internet: On in notebook settings, downloads about 2.3 GB once.


In [4]:
"""
Sanity check only: 1 epoch, 10% of the training data, both model families.
This proves the pipeline runs end to end before Phase 2 commits real GPU hours.
"""
from ultralytics import YOLO


def sanity_check(model_name: str, data_yaml: str = "VisDrone.yaml") -> None:
    try:
        model = YOLO(model_name)
        model.train(
            data=data_yaml,
            epochs=1,
            imgsz=640,
            fraction=0.1,   # 10% of train data, enough to catch errors without waiting on the full set
            device=0,
            project="runs/phase1_sanity",
            name=model_name.replace(".pt", ""),
            verbose=True,
        )
        print(f"{model_name}: sanity check passed, pipeline runs end to end.")
    except Exception as e:
        # surface which model failed, a bare traceback from inside ultralytics won't say that on its own
        raise RuntimeError(f"{model_name} sanity check failed: {e}") from e


for name in ["yolo26n.pt", "yolo11n.pt"]:
    sanity_check(name)

Ultralytics 8.4.124 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=VisDrone.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=0.1, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo26n, nbs=64, nms=False, opset=None, optimize

In [5]:
"""Confirm both sanity runs actually wrote weights to disk."""
from pathlib import Path

runs_dir = Path("runs/phase1_sanity")
if not runs_dir.exists():
    raise RuntimeError("runs/phase1_sanity does not exist, the sanity check cell above did not run or failed silently.")

for run in sorted(runs_dir.iterdir()):
    weights = run / "weights" / "last.pt"
    status = "OK" if weights.exists() else "MISSING"
    print(f"{run.name}: {status}")

RuntimeError: runs/phase1_sanity does not exist, the sanity check cell above did not run or failed silently.

## Phase 1 is complete when:
- Both sanity check runs print "pipeline runs end to end"
- Both `runs/phase1_sanity/<model>/weights/last.pt` files show `OK` above

**Next: Phase 2**, full training run, both models, same epoch budget, no `fraction` limit.